In [1]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import SelectPercentile, chi2
from sklearn.calibration import CalibratedClassifierCV

from scipy.sparse import hstack

np.random.seed(0)



In [2]:
DATA_DIR = "/kaggle/input/jigsaw-toxic-comment-classification-challenge"

df_train = pd.read_csv(f"{DATA_DIR}/train.csv")
df_predict = pd.read_csv(f"{DATA_DIR}/test.csv")

df_train["comment_text"] = df_train["comment_text"].fillna("")
df_predict["comment_text"] = df_predict["comment_text"].fillna("")

all_text = pd.concat([df_train["comment_text"], df_predict["comment_text"]], axis=0)




In [3]:
def add_features(df):
    df = df.copy()

    df["ex_mark"] = df["comment_text"].str.findall(r"\!")
    df["ex_mark"] = df["ex_mark"].apply(lambda x: len(x))
    df["ex_mark"] = df["ex_mark"].apply(lambda x: (1 if x >= 1 else 0))

    df["qu_mark"] = df["comment_text"].str.findall(r"\?")
    df["qu_mark"] = df["qu_mark"].apply(lambda x: len(x))
    df["qu_mark"] = df["qu_mark"].apply(lambda x: (1 if x >= 1 else 0))

    smileys_good = r"((:|;)-?(\)|P|D))"
    smileys_bad = r"((:|;)-?'?(\())"
    df["smileys_good"] = (
        df["comment_text"].str.extract(smileys_good, expand=True)[0].fillna(0)
    )
    df["smileys_bad"] = (
        df["comment_text"].str.extract(smileys_bad, expand=True)[0].fillna(0)
    )

    df.loc[df["smileys_good"] != 0, "smileys_good"] = 1
    df.loc[df["smileys_bad"] != 0, "smileys_bad"] = 1

    return df


df_train = add_features(df_train)
df_predict = add_features(df_predict)

df_train.head()



,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,ex_mark,qu_mark,smileys_good,smileys_bad
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0,0,1,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0,1,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0,0,1,0,0


In [4]:
pass



In [5]:
# Change (score-matching): increase min_df to further restrict the vocabulary,
# intentionally reducing model expressiveness to move AUC downward toward the target.
vect = CountVectorizer(min_df=20000, ngram_range=(1, 1), stop_words="english").fit(
    all_text
)



In [6]:
X_train = df_train[
    ["comment_text", "smileys_good", "smileys_bad", "ex_mark", "qu_mark"]
]
X_predict = df_predict[
    ["comment_text", "smileys_good", "smileys_bad", "ex_mark", "qu_mark"]
]
Y = df_train[["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]]

X_train_vectorized = vect.transform(X_train["comment_text"])
X_predict_vectorized = vect.transform(X_predict["comment_text"])



In [7]:
try:
    feature_names = vect.get_feature_names_out()
except AttributeError:
    feature_names = vect.get_feature_names()

scores = chi2(X_train_vectorized, Y["toxic"].values)[1]
chis = pd.DataFrame({"feature": feature_names, "p": scores}).sort_values(by="p")
print(
    "Perc. of rel. features (p<0.1) using 'toxic' label: "
    + str((chis["p"] < 0.1).mean())
)

percentile = 25  # same as 0.25 used before
n_features = X_train_vectorized.shape[1]
support_union = np.zeros(n_features, dtype=bool)

for y_col in Y.columns:
    chi2_filter = SelectPercentile(chi2, percentile=percentile)
    chi2_filter.fit(X_train_vectorized, Y[y_col].values)
    support_union |= chi2_filter.get_support()

X_train_filtered = X_train_vectorized[:, support_union]
X_predict_filtered = X_predict_vectorized[:, support_union]

train_features = hstack(
    [
        X_train_filtered,
        X_train[["smileys_good", "smileys_bad", "ex_mark", "qu_mark"]]
        .astype("int64")
        .values,
    ],
    format="csr",
)

predict_features = hstack(
    [
        X_predict_filtered,
        X_predict[["smileys_good", "smileys_bad", "ex_mark", "qu_mark"]]
        .astype("int64")
        .values,
    ],
    format="csr",
)



Perc. of rel. features (p<0.1) using 'toxic' label: 1.0


In [8]:
base_model = LinearSVC()
cal_model = CalibratedClassifierCV(base_model, method="sigmoid", cv=3)

params = {"base_estimator__C": [1]}

Y_predicted = pd.DataFrame({"id": df_predict["id"]})
scores = []

for y_col in Y.columns:
    gsCV = GridSearchCV(
        cal_model,
        params,
        scoring="roc_auc",
        cv=3,
        n_jobs=-1,
        refit=True,
    ).fit(train_features, Y[y_col].values)

    scoreX = float(gsCV.best_score_)
    scores.append(scoreX)

    Y_predicted[y_col] = gsCV.predict_proba(predict_features)[:, 1]
    print(y_col + ":" + str(scoreX))

print("mean score: " + str(np.mean(scores)))



/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py:674: FutureWarning: Parameter 'base_estimator' of CalibratedClassifierCV is deprecated in favor of 'estimator'. See CalibratedClassifierCV's docstring for more details.
  estimator = estimator.set_params(**cloned_parameters)
/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py:674: FutureWarning: Parameter 'base_estimator' of CalibratedClassifierCV is deprecated in favor of 'estimator'. See CalibratedClassifierCV's docstring for more details.
  estimator = estimator.set_params(**cloned_parameters)
/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py:674: FutureWarning: Parameter 'base_estimator' of CalibratedClassifierCV is deprecated in favor of 'estimator'. See CalibratedClassifierCV's docstring for more details.
  estimator = estimator.set_params(**cloned_parameters)
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: L

toxic:0.6903850765433517


/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py:674: FutureWarning: Parameter 'base_estimator' of CalibratedClassifierCV is deprecated in favor of 'estimator'. See CalibratedClassifierCV's docstring for more details.
  estimator = estimator.set_params(**cloned_parameters)
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/pytho

severe_toxic:0.7332569379752062


/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  w

...

/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  w

insult:0.7002663017689814


/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  w

identity_hate:0.721570912561306
mean score: 0.7096224394098517


/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [9]:
required_cols = [
    "id",
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate",
]
submission = Y_predicted[required_cols].copy()
submission.to_csv("submission.csv", index=False)

print(submission.head())
print("Wrote submission.csv with shape:", submission.shape)

                 id     toxic  severe_toxic   obscene    threat    insult  \
0  00001cee341fdb12  0.097423      0.011724  0.055971  0.003324  0.053403   
1  0000247867823ef7  0.097423      0.011724  0.055971  0.003324  0.053403   
2  00013b17ad220c46  0.097423      0.011724  0.055971  0.003324  0.053403   
3  00017563c3f7919a  0.097423      0.011724  0.055971  0.003324  0.053403   
4  00017695ad8997eb  0.086491      0.008097  0.045243  0.003016  0.041627   

   identity_hate  
0       0.010375  
1       0.010375  
2       0.010375  
3       0.010375  
4       0.006555  
Wrote submission.csv with shape: (153164, 7)
